In [1]:
import numpy as np
from mesh.mesh_read_plot3D import StructuredMeshInitialization2D
from mesh.mesh import MeshGeoCalculator2D
import boundary.boundary as bd
import type_transform as tf
import config
import Initialization as initial
from solver.solver import CFDSolver
from post_output.output_tecplot import output_forces
from post_output.output_tecplot import output_tecplot_series
import pickle
import time
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

def merge_blocks_geo(blocks):
    geo_list = []
    split_indices = []
    ni_offset = 0

    for blk in blocks:
        geo = blk.geo[..., 0:2]  # 只取 X, Y
        ni = geo.shape[0]
        geo_list.append(geo)
        split_indices.append((ni_offset, ni_offset + ni))
        ni_offset += ni

    geo_all = np.concatenate(geo_list, axis=0)  # shape: (ni_total, nj, 4)
    return geo_all, split_indices


def merge_blocks_fluid(blocks):
    fluid_list = []

    for blk in blocks:
        w = blk.fluid
        fluid_list.append(w)

    fluid_all = np.concatenate(fluid_list, axis=0)  # shape: (ni_total, nj, 4)
    return fluid_all


def split_by_indices(w_all, split_indices):
    parts_list = []
    for start, end in split_indices:
        parts_list.append(w_all[start:end])
    return parts_list

# 读取网格和边界条件，预处理网格

In [2]:
mesh_read = StructuredMeshInitialization2D()
mesh_read.load_file("RAE2822.grd", "RAE2822.inp", 1)   # 网格尺寸缩放倍率
mesh_read.merge_blocks_2D()
mesh_read.interface_transform_cal()
mesh_read.print_block_info()

[Block 0]  shape: (33, 65, 1)
  - BC type -1, source [1, 33, 1, 1], target_block 2, target [33, 1, 1, 1], transform (-1, -2)
  - BC type -1, source [33, 33, 1, 65], target_block 1, target [1, 1, 1, 65], transform (1, 2)
  - BC type 4, source (1, 33, 65, 65), target_block N/A, target N/A
  - BC type 4, source (1, 1, 1, 65), target_block N/A, target N/A
[Block 1]  shape: (305, 65, 1)
  - BC type 2, source (1, 305, 1, 1), target_block N/A, target N/A
  - BC type -1, source [305, 305, 1, 65], target_block 2, target [1, 1, 1, 65], transform (1, 2)
  - BC type 4, source (1, 305, 65, 65), target_block N/A, target N/A
  - BC type -1, source [1, 1, 1, 65], target_block 0, target [33, 33, 1, 65], transform (1, 2)
[Block 2]  shape: (33, 65, 1)
  - BC type -1, source [33, 1, 1, 1], target_block 0, target [1, 33, 1, 1], transform (-1, -2)
  - BC type 4, source (33, 33, 1, 65), target_block N/A, target N/A
  - BC type 4, source (1, 33, 65, 65), target_block N/A, target N/A
  - BC type -1, source [1,

# 计算网格几何参数(格心坐标，面向量，面积）

In [3]:
mesh_geocal = MeshGeoCalculator2D(mesh_read)
mesh_geocal.compute_centroids()
mesh_geocal.compute_volumes()
mesh_geocal.compute_face_vectors()
blocks = np.copy(mesh_geocal.mesh.blocks)
bd.crate_ghost_cells(blocks, config.GHOST_LAYER, config.N_C)
blocks_cal = tf.trans_list2numpy_2d(blocks, config.N_C)

[Block 0] xc range: (-6.60147775, 23.80002500), yc range: (-24.57599500, -0.00010715)
[Block 1] xc range: (-18.24956750, 1.63837900), yc range: (-19.32460500, 20.49909000)
[Block 2] xc range: (-1.38759415, 23.85551750), yc range: (-1.43959700, 21.87501000)
[Block 0] volume range: (1.01809356e-08, 3.15633363e+01)
[Block 1] volume range: (9.85525067e-09, 3.09754767e+00)
[Block 2] volume range: (1.01832477e-08, 3.03117921e+01)


# 初始化流场和边界条件

In [4]:
initial.initialization_from_farfield(blocks_cal)

# 迭代计算

In [6]:
cfdsolver = CFDSolver(blocks_cal, config.GAMMA, 0.5)

# 时间离散格式
cfdsolver.temporal_discrete = 1

# 是否当地时间步长
cfdsolver.if_localdt = 1

# 是否生成一个沿时间序列的大矩阵
cfdsolver.if_output_npy = 0

#创建历史记录文件
with open("history.dat", "w") as f:
    f.write("Iter\tResidual\tFx\tFy\tTime(s)\n")

start_time = time.time()

for _ in range(10000):

    cfdsolver.rk4_iterate()
    res_norm = cfdsolver.compute_global_residual_norm()
    cfdsolver.residuals.append(res_norm)

    if cfdsolver.iteration % 10 == 0:
        fx, fy = output_forces(cfdsolver.blocks)

        # 记录从上一次10次迭代起的时间
        elapsed = time.time() - start_time
        start_time = time.time()  # 重置计时器

        print(f"[Iter {cfdsolver.iteration}] Residual = {res_norm:.3e}")
        print(f"Forces = {fx:.6f}, {fy:.6f}")
        print(f"Time for last 10 iters = {elapsed:.2f} s")

        with open("history.dat", "a") as f:
            f.write(f"{cfdsolver.iteration}\t\t\t{res_norm:.6e}\t\t\t\t{fx:.6f}\t\t\t{fy:.6f}\t\t\t{elapsed:.2f}\n")

    os.makedirs("results", exist_ok=True)
    if cfdsolver.iteration % 100 == 0:
        tecplot_filename = os.path.join("results", f"solution_iter_{cfdsolver.iteration}.dat")
        pkl_filename = os.path.join("results", f"blocks_result_iter_{cfdsolver.iteration}.pkl")
        output_tecplot_series(cfdsolver.blocks, cfdsolver.iteration, tecplot_filename)
        with open(pkl_filename, 'wb') as f:
            pickle.dump(cfdsolver.blocks, f)

    if res_norm < 1e-8:
        print("收敛达到停止条件")
        break

[Iter 10] Residual = 4.270e-02
Forces = 3664.218649, 4192.477983
Time for last 10 iters = 23.97 s
[Iter 20] Residual = 4.335e-02
Forces = 3865.452895, 4193.924403
Time for last 10 iters = 6.71 s
[Iter 30] Residual = 3.255e-02
Forces = 2905.562506, 3536.911998
Time for last 10 iters = 6.56 s
[Iter 40] Residual = 3.732e-02
Forces = 3483.968240, 3883.083294
Time for last 10 iters = 6.59 s
[Iter 50] Residual = 2.740e-02
Forces = 3600.447498, 3911.926955
Time for last 10 iters = 6.60 s


KeyboardInterrupt: 